In [1]:
from __future__ import print_function
import sklearn
import numpy as np
import sklearn
import sklearn.ensemble
import sklearn.metrics

# Importing LimeTextExplainer (aix360 sytle)
from mindxlib.post_hoc_explainer import LimeTextExplainer

In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
from sklearn.datasets import fetch_20newsgroups
newsgroups_train = fetch_20newsgroups(subset='train')
newsgroups_test = fetch_20newsgroups(subset='test')
# making class names shorter
class_names = [x.split('.')[-1] if 'misc' not in x else '.'.join(x.split('.')[-2:]) for x in newsgroups_train.target_names]
class_names[3] = 'pc.hardware'
class_names[4] = 'mac.hardware'

In [4]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(lowercase=False)
train_vectors = vectorizer.fit_transform(newsgroups_train.data)
test_vectors = vectorizer.transform(newsgroups_test.data)

In [5]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB(alpha=.01)
nb.fit(train_vectors, newsgroups_train.target)

MultinomialNB(alpha=0.01)

In [6]:
pred = nb.predict(test_vectors)
sklearn.metrics.f1_score(newsgroups_test.target, pred, average='weighted')

0.8350184193998174

In [7]:
from sklearn.pipeline import make_pipeline
c = make_pipeline(vectorizer, nb)

In [8]:
print(c.predict_proba([newsgroups_test.data[0]]).round(3))

[[0.001 0.01  0.003 0.047 0.006 0.002 0.003 0.521 0.022 0.008 0.025 0.
  0.331 0.003 0.006 0.    0.003 0.    0.001 0.009]]


In [9]:
limeexplainer = LimeTextExplainer(class_names=class_names)
# print(type(limeexplainer))

In [10]:
idx = 1340
# Explain input instances
exp = limeexplainer.explain_instance(newsgroups_test.data[idx], c.predict_proba, num_features=6, labels=[0, 17])
print('Document id: %d' % idx)
print('Predicted class =', class_names[nb.predict(test_vectors[idx]).reshape(1,-1)[0,0]])
print('True class: %s' % class_names[newsgroups_test.target[idx]])

Document id: 1340
Predicted class = atheism
True class: atheism


In [11]:
print ('Explanation for class %s' % class_names[0])
print ('\n'.join(map(str, exp.as_list(label=0))))
print ()
print ('Explanation for class %s' % class_names[17])
print ('\n'.join(map(str, exp.as_list(label=17))))

Explanation for class atheism
('Caused', 0.2590648109007334)
('Rice', 0.14844279659663456)
('Genocide', 0.1260494415184468)
('owlnet', -0.09080385500770491)
('Semitic', -0.08599538944895356)
('scri', -0.0835955495105149)

Explanation for class mideast
('fsu', -0.055875066111175216)
('Theism', -0.052613901614870937)
('Luther', -0.051764698609399806)
('Caused', -0.03478373599191443)
('jews', 0.03442185772746421)
('PBS', 0.03405528112257664)
